# Fivetran + Fabric: V-Order Protection & Protocol Management

## Overview

This notebook provides essential maintenance scripts for managing Fivetran connectors writing to Microsoft Fabric lakehouses. It addresses the critical incompatibility between Fabric's V-Order optimization and Fivetran's Delta Lake writer capabilities.

**Purpose:**
- Diagnose Delta Lake protocol version issues
- Disable V-Order optimization on Fivetran-managed tables
- Monitor protocol versions to prevent sync failures
- Validate and restore tables after Fivetran re-sync

**When to Use:**
- **Preventive:** Run immediately after setting up new Fivetran connectors
- **Monitoring:** Schedule Cell 3 to run weekly
- **Reactive:** Use when Fivetran reports "Invalid protocol version" errors

---

## The Problem

### Delta Lake Protocol Compatibility

| Component | Reader Version | Writer Version |
|-----------|---------------|----------------|
| Fivetran Delta Standalone | 1 | 2 |
| Fabric V-Order | 3 | 7 |

**Issue:** When Fabric's V-Order optimization runs on tables, it upgrades them to protocol writer version 7. Fivetran only supports writer version 2, causing sync failures.

**Key Facts:**
- Protocol upgrades are **irreversible**
- V-Order is often enabled by default in Fabric
- Once upgraded, tables must be deleted and recreated to restore Fivetran compatibility

**Solution:** Explicitly disable V-Order on all Fivetran-managed tables and monitor for drift.

---

## Cell 1: Protocol Version Audit

**Purpose:** Scan all tables in the lakehouse and identify protocol version incompatibilities.

**What it does:**
1. Lists all tables in the current lakehouse
2. Checks each table's Delta Lake protocol version (reader and writer)
3. Checks V-Order setting for each table
4. Flags tables that are incompatible with Fivetran (writer v > 2 or reader v > 1)

**Output:**
- Status for each table (✓ Compatible or ❌ INCOMPATIBLE)
- Protocol versions (Writer and Reader)
- V-Order setting status
- Summary list of incompatible tables

**When to run:**
- Initial diagnosis when Fivetran reports errors
- After setting up new lakehouses
- Periodic audits to verify configuration

**Expected results:**
- ✓ Compatible: Writer v1-2, Reader v1
- ❌ INCOMPATIBLE: Writer v3+, Reader v2+ (requires fix)

In [ ]:
# Cell 1: Protocol Version Audit

# Get all tables
tables = spark.sql("SHOW TABLES").collect()

print("=" * 70)
print("PROTOCOL VERSION AUDIT")
print("=" * 70)

incompatible_tables = []

for table in tables:
    schema = getattr(table, 'database', None) or getattr(table, 'namespace', None)
    table_name = table.tableName
    full_name = f"`{schema}`.`{table_name}`" if schema else f"`{table_name}`"
    
    try:
        detail = spark.sql(f"DESCRIBE DETAIL {full_name}").collect()[0]
        
        writer_v = detail.minWriterVersion
        reader_v = detail.minReaderVersion
        vorder = detail.properties.get('delta.parquet.vorder.enabled', 'not set')
        
        # Flag incompatible tables
        if writer_v > 2 or reader_v > 1:
            status = "❌ INCOMPATIBLE"
            incompatible_tables.append(full_name)
        else:
            status = "✓ Compatible"
        
        print(f"{status} - {full_name}")
        print(f"   Protocol: Writer v{writer_v}, Reader v{reader_v}")
        print(f"   V-Order: {vorder}")
        print()
        
    except Exception as e:
        print(f"Error: {full_name} - {e}\n")

print("=" * 70)
if incompatible_tables:
    print(f"⚠️ FOUND {len(incompatible_tables)} INCOMPATIBLE TABLES:")
    for t in incompatible_tables:
        print(f"  - {t}")
else:
    print("✓ All tables compatible with Fivetran")
print("=" * 70)

## Cell 2: Disable V-Order for All Tables (Prevention)

**Purpose:** Protect all tables in the lakehouse from protocol upgrades by disabling V-Order optimization.

**What it does:**
1. Iterates through all tables in the lakehouse
2. Sets table properties to disable V-Order, deletion vectors, and change data feed
3. Reports success/failure for each table

**Table properties set:**
```sql
'delta.parquet.vorder.enabled' = 'false'         -- Disables V-Order
'delta.enableDeletionVectors' = 'false'          -- Disables deletion vectors (protocol v7)
'delta.enableChangeDataFeed' = 'false'           -- Disables CDC
```

**When to run:**
- **Immediately** after setting up new Fivetran connectors
- After creating a new lakehouse that will receive Fivetran data
- Any time new tables are added to the lakehouse

**Important notes:**
- Safe to run multiple times (idempotent)
- Does not affect existing data or queries
- Only changes table metadata/properties
- Should be run on **staging/landing lakehouses only** (not enriched lakehouses where you want V-Order)

In [ ]:
# Cell 2: Disable V-Order for All Tables

tables = spark.sql("SHOW TABLES").collect()

print("Disabling V-Order optimization for all tables...")
print("=" * 70)

for table in tables:
    schema = getattr(table, 'database', None) or getattr(table, 'namespace', None)
    table_name = table.tableName
    full_name = f"`{schema}`.`{table_name}`" if schema else f"`{table_name}`"
    
    try:
        spark.sql(f"""
            ALTER TABLE {full_name}
            SET TBLPROPERTIES (
                'delta.parquet.vorder.enabled' = 'false',
                'delta.enableDeletionVectors' = 'false',
                'delta.enableChangeDataFeed' = 'false'
            )
        """)
        
        print(f"✓ Protected: {full_name}")
        
    except Exception as e:
        print(f"❌ Error: {full_name} - {e}")

print("=" * 70)
print("✓ V-Order disabled for all tables")

## Cell 3: Protocol Version Monitoring (Scheduled)

**Purpose:** Detect protocol version drift and alert when tables have been upgraded.

**What it does:**
1. Scans all tables for protocol version changes
2. Identifies tables that have drifted from expected protocol v2
3. Reports violations with specific version numbers
4. Provides a clear pass/fail status

**How to schedule:**
1. Create a Fabric Data Pipeline
2. Add a "Notebook" activity
3. Select this notebook and specify Cell 3
4. Set recurrence: Weekly (recommended) or Daily
5. Configure failure notifications (email, Teams, etc.)

**Integration with alerting:**
- Replace the `# TODO` comment with your notification logic
- Examples: Teams webhook, email via Logic Apps, ServiceNow incident

**When to run:**
- **Scheduled:** Weekly via Data Pipeline (recommended)
- **Ad-hoc:** Before major releases or migrations
- **Reactive:** When investigating Fivetran issues

**Response to alerts:**
1. Identify what caused the upgrade (Fabric job, manual OPTIMIZE, etc.)
2. Follow resolution steps if Fivetran is affected
3. Re-run Cell 2 to re-apply protection

In [ ]:
# Cell 3: Protocol Version Monitoring
# Schedule this to run weekly via Data Pipelines

from datetime import datetime

print(f"Protocol Version Monitoring - {datetime.now()}")
print("=" * 70)

tables = spark.sql("SHOW TABLES").collect()
alerts = []

for table in tables:
    schema = getattr(table, 'database', None) or getattr(table, 'namespace', None)
    table_name = table.tableName
    full_name = f"`{schema}`.`{table_name}`" if schema else f"`{table_name}`"
    
    try:
        detail = spark.sql(f"DESCRIBE DETAIL {full_name}").collect()[0]
        
        if detail.minWriterVersion > 2 or detail.minReaderVersion > 1:
            alerts.append({
                'table': full_name,
                'writer': detail.minWriterVersion,
                'reader': detail.minReaderVersion
            })
    except:
        pass

if alerts:
    print("⚠️ PROTOCOL VERSION DRIFT DETECTED:")
    print("=" * 70)
    for alert in alerts:
        print(f"TABLE: {alert['table']}")
        print(f"  Writer: v{alert['writer']} (expected: v2)")
        print(f"  Reader: v{alert['reader']} (expected: v1)")
        print()
    
    # TODO: Add notification logic (email, Teams webhook, etc.)
    # Example: Send email to data-engineering@company.com
    
else:
    print("✓ All tables at protocol v2 - No action needed")

print("=" * 70)

## Cell 4: Delete Files for Protocol Reset (Use with Caution)

**Purpose:** Delete underlying Delta Lake files to prepare for Fivetran re-sync and protocol reset.

### ⚠️ CRITICAL - READ BEFORE RUNNING:

- This cell **deletes data files** from specified tables
- Tables will be temporarily empty/unavailable
- Downstream processes (shortcuts, views, reports) will fail until Fivetran re-syncs
- **ALWAYS run in preview mode first** (`DELETE_FILES = False`)
- Only use when Fivetran reports "Invalid protocol version" errors

**What it does:**
1. Lists all files in specified table directories
2. Shows file count and total size for each table
3. In preview mode: Shows what would be deleted
4. In delete mode: Removes all Delta files to reset protocol version

**How to use:**
1. Update `tables_to_fix` list with affected table names from Fivetran error
2. Run with `DELETE_FILES = False` to preview
3. Review output carefully
4. Change `DELETE_FILES = True` and run again to actually delete
5. Immediately trigger Fivetran full historical re-sync

**Prerequisites:**
- Document current row counts for validation
- Communicate maintenance window to stakeholders
- Pause downstream pipelines/refreshes
- Coordinate with Fivetran team

**After running:**
- Tables will show 0 rows or errors when queried
- Table metadata/schema remains intact
- Fivetran will repopulate data during re-sync
- Run Cell 5 after Fivetran completes

**Alternative:** You can also delete files manually via OneLake File Explorer:
- Navigate to `Files/Tables/[table_name]/`
- Delete all contents (keep folder structure)

In [ ]:
# Cell 4: Delete Files for Protocol Reset
# BE CAREFUL - THIS DELETES DATA

from notebookutils import mssparkutils

# List affected tables from Fivetran error message
tables_to_fix = [
    'schema__table1',
    'schema__table2',
    # Add all affected tables
]

# Preview mode - shows what will be deleted
DELETE_FILES = False  # Set to True to actually delete

print("=" * 70)
print("FILE DELETION PREVIEW" if not DELETE_FILES else "DELETING FILES")
print("=" * 70)

for table in tables_to_fix:
    print(f"\n📁 {table}:")
    
    try:
        # Get table location
        detail = spark.sql(f"DESCRIBE DETAIL {table}").collect()[0]
        table_location = detail.location
        files = mssparkutils.fs.ls(table_location)
        
        total_size = sum([f.size for f in files]) / (1024**3)  # Convert to GB
        
        print(f"   Location: {table_location}")
        print(f"   Files to delete: {len(files)}")
        print(f"   Total size: {total_size:.2f} GB")
        
        if DELETE_FILES:
            for f in files:
                mssparkutils.fs.rm(f.path, recurse=True)
            print(f"   ✓ Deleted all files")
        else:
            print(f"   (preview mode - no files deleted)")
            
    except Exception as e:
        print(f"   ❌ Error: {e}")

if not DELETE_FILES:
    print("\n⚠️ Set DELETE_FILES = True to actually delete files")
else:
    print("\n✅ File deletion complete")
    print("Next step: Trigger Fivetran full historical re-sync")

## Cell 5: Re-Apply V-Order Protection After Re-Sync

**Purpose:** Protect recreated tables from future protocol upgrades.

**When to run:**
- **Immediately** after Fivetran completes full historical re-sync
- This ensures the freshly created tables are protected from V-Order

**What it does:**
1. Takes the list of tables that were just recreated
2. Applies V-Order protection properties to each table
3. Confirms protection was successfully applied

**How to use:**
1. Update `recreated_tables` list with the same tables from Cell 4
2. Run this cell immediately after Fivetran sync completes
3. Verify all tables show "✓ Protected"

**Why this is necessary:**
- When Fivetran recreates tables, they start with default Fabric settings
- The lakehouse may have V-Order enabled by default
- This cell ensures the new tables are explicitly protected

In [ ]:
# Cell 5: Re-Apply V-Order Protection After Re-Sync
# Run immediately after Fivetran completes re-sync

recreated_tables = [
    'schema__table1',
    'schema__table2',
    # Add all tables that were just recreated
]

print("Re-applying V-Order protection to recreated tables...")
print("=" * 70)

for table in recreated_tables:
    try:
        spark.sql(f"""
            ALTER TABLE {table}
            SET TBLPROPERTIES (
                'delta.parquet.vorder.enabled' = 'false',
                'delta.enableDeletionVectors' = 'false'
            )
        """)
        print(f"✓ Protected: {table}")
    except Exception as e:
        print(f"❌ Error: {table} - {e}")

print("=" * 70)
print("✓ All tables protected")

## Cell 6: Final Validation

**Purpose:** Comprehensive validation that tables are fully restored and protected.

**What it validates:**
1. **Row counts:** Confirms data was successfully restored
2. **Protocol versions:** Verifies tables are back at protocol v2/v1
3. **V-Order setting:** Confirms protection is applied

**When to run:**
- After completing the full restoration process (Cells 4 & 5)
- Before resuming normal operations and downstream pipelines

**How to use:**
1. Update `tables_to_validate` with the same list from Cells 4 & 5
2. Run this cell
3. Compare row counts to your pre-fix documentation
4. Verify all tables show "✓" status

**Expected output:**
```
✓ table_name:
   Rows: [count matching pre-fix]
   Protocol: W2/R1
   V-Order: false
```

**If validation fails:**
- ❌ STILL UPGRADED: Protocol not reset - contact Fivetran support
- ❌ V-ORDER ENABLED: Re-run Cell 5
- Row count mismatch: Investigate Fivetran sync logs

**After successful validation:**
1. Test downstream shortcuts and views
2. Resume scheduled pipelines
3. Refresh dependent materialized views
4. Notify stakeholders that maintenance is complete

In [ ]:
# Cell 6: Final Validation
# Confirms tables are restored and protected

tables_to_validate = [
    'schema__table1',
    'schema__table2',
    # Add all restored tables
]

print("=" * 70)
print("POST-RESTORATION VALIDATION")
print("=" * 70)

all_good = True

for table in tables_to_validate:
    try:
        # Check row count
        count = spark.sql(f"SELECT COUNT(*) FROM {table}").collect()[0][0]
        
        # Check protocol
        detail = spark.sql(f"DESCRIBE DETAIL {table}").collect()[0]
        writer_v = detail.minWriterVersion
        reader_v = detail.minReaderVersion
        vorder = detail.properties.get('delta.parquet.vorder.enabled', 'not set')
        
        # Validate
        if writer_v > 2 or reader_v > 1:
            status = "❌ STILL UPGRADED"
            all_good = False
        elif vorder == 'true':
            status = "❌ V-ORDER ENABLED"
            all_good = False
        else:
            status = "✓"
        
        print(f"{status} {table}:")
        print(f"   Rows: {count:,}")
        print(f"   Protocol: W{writer_v}/R{reader_v}")
        print(f"   V-Order: {vorder}")
        print()
        
    except Exception as e:
        print(f"❌ {table}: ERROR - {e}\n")
        all_good = False

print("=" * 70)
if all_good:
    print("✅ ALL TABLES SUCCESSFULLY RESTORED")
    print("Fivetran can now write to these tables")
    print("\nNext steps:")
    print("1. Test downstream shortcuts and views")
    print("2. Resume scheduled pipelines")
    print("3. Refresh dependent materialized views")
else:
    print("⚠️ ISSUES DETECTED - Review output above")
    print("\nTroubleshooting:")
    print("- STILL UPGRADED: Contact Fivetran support")
    print("- V-ORDER ENABLED: Re-run Cell 5")
print("=" * 70)

---

## Quick Reference Guide

### Preventive Maintenance (New Connector Setup)
1. **Run Cell 2** immediately after Fivetran connector setup
2. **Schedule Cell 3** to run weekly via Data Pipeline

### Reactive Maintenance (Protocol Version Error)
1. **Run Cell 1** to identify affected tables
2. Document current row counts
3. **Run Cell 4** (preview first, then delete)
4. Trigger Fivetran full historical re-sync
5. **Run Cell 5** immediately after sync completes
6. **Run Cell 6** to validate restoration

### Regular Monitoring
- **Run Cell 1** monthly for comprehensive audit
- **Run Cell 3** weekly via scheduled pipeline

---

## Best Practices

### Do's
- ✅ Run Cell 2 on all new Fivetran-managed lakehouses
- ✅ Schedule Cell 3 for weekly monitoring
- ✅ Use V-Order freely in downstream enriched lakehouses
- ✅ Document maintenance windows before running Cell 4

### Don'ts
- ❌ Never enable V-Order on Fivetran-managed tables
- ❌ Don't run `OPTIMIZE table VORDER` on staging tables
- ❌ Don't skip validation (Cell 6) after restoration

---

## Additional Resources

- [Fivetran Delta Lake Connector Docs](https://fivetran.com/docs/destinations/delta-lake)
- [Delta Lake Protocol Versioning](https://docs.delta.io/latest/versioning.html)
- [Microsoft Fabric V-Order](https://learn.microsoft.com/en-us/fabric/data-engineering/delta-optimization-and-v-order)

---

**Document Information:**
- **Last Updated:** December 2024
- **Author:** Data Engineering Team
- **Version:** 1.0